# Task 4: MongoDB and PyMongo

This notebook imports original and cleaned BugRadar issue documents into separate MongoDB collections, demonstrates safe Create, Read, Update, and Delete operations, and runs useful MongoDB queries. Both JSON files remain read-only.


## 4.1 Imports and MongoDB Connection

In [13]:
import hashlib
import json
from collections import Counter
from pathlib import Path
from pprint import pprint

import pymongo
from pymongo import ReplaceOne, ASCENDING
from pymongo.errors import PyMongoError

MONGODB_URI = 'mongodb://localhost:27017/'
DATABASE_NAME = 'bugradar_db'

# MongoClient connects lazily; ping forces an actual check with a bounded timeout.
mongo_client = pymongo.MongoClient(
    MONGODB_URI,
    serverSelectionTimeoutMS=5000,
    connectTimeoutMS=5000,
)
try:
    ping_result = mongo_client.admin.command('ping')
except PyMongoError as connection_error:
    mongo_client.close()
    raise RuntimeError('MongoDB is not reachable at localhost:27017. Start the local service and rerun this cell.') from connection_error
if ping_result.get('ok') != 1:
    mongo_client.close()
    raise RuntimeError(f'MongoDB ping returned an unexpected response: {ping_result!r}')

database = mongo_client[DATABASE_NAME]
print(f'MongoDB ping successful. Using only {DATABASE_NAME!r}.')

MongoDB ping successful. Using only 'bugradar_db'.


## 4.2 Load and Validate Both JSON Files


In [14]:
DATA_FILES = {
    'raw': Path('data') / 'bugradar issues og.json',
    'cleaned': Path('data') / 'bugradar_issues_cleaned.json',
}
project_root = next(
    (folder for folder in (Path.cwd(), *Path.cwd().parents)
     if all((folder / relative_path).is_file() for relative_path in DATA_FILES.values())),
    None,
)
if project_root is None:
    raise FileNotFoundError(f'Could not locate both JSON files above {Path.cwd()}.')

def load_issue_documents(json_path):
    """Load a UTF-8 JSON list and verify it contains only document dictionaries."""
    try:
        with json_path.open('r', encoding='utf-8') as input_file:
            documents = json.load(input_file)
    except (OSError, UnicodeDecodeError, json.JSONDecodeError) as file_error:
        raise ValueError(f'Could not load valid UTF-8 JSON from {json_path}: {file_error}') from file_error
    if not isinstance(documents, list) or not all(isinstance(document, dict) for document in documents):
        raise ValueError(f'{json_path} must contain a JSON list of document dictionaries.')
    return documents

raw_path = project_root / DATA_FILES['raw']
cleaned_path = project_root / DATA_FILES['cleaned']
raw_documents = load_issue_documents(raw_path)
cleaned_documents = load_issue_documents(cleaned_path)
if not raw_documents or not cleaned_documents:
    raise ValueError('Both issue JSON files must contain at least one document.')
print(f'Project root: {project_root}')
print(f'Original documents: {len(raw_documents):,}; cleaned documents: {len(cleaned_documents):,}')
print('One original document:')
pprint(raw_documents[0], sort_dicts=False, width=100)
print('One cleaned document:')
pprint(cleaned_documents[0], sort_dicts=False, width=100)

Project root: c:\Users\riya choudhary\OneDrive\Desktop\BugRadar
Original documents: 5,000; cleaned documents: 5,000
One original document:
{'id': 1000000,
 'number': 1,
 'title': 'Example missing for refresh the page',
 'body': None,
 'issue_category': 'documentation',
 'state': 'open',
 'state_reason': None,
 'priority': 'medium',
 'created_at': '2025-08-05T17:32:00Z',
 'closed_at': None,
 'resolution_hours': None,
 'labels': [],
 'comments': 7,
 'user': {'login': None, 'type': None},
 'assignees': [],
 'repository': {'name': 'auth-api', 'language': 'TypeScript', 'type': 'backend', 'stars': 144},
 'environment': {'platform': 'Firefox 131', 'version': '1.8.0'},
 'milestone': {'title': 'v2.0'},
 'url': 'https://example.invalid/auth-api/issues/1',
 'source': 'synthetic_educational_dataset'}
One cleaned document:
{'id': 1000000,
 'number': 1,
 'title': 'Example missing for refresh the page',
 'body': None,
 'issue_category': 'documentation',
 'state': 'open',
 'state_reason': None,
 'prio

In [16]:
nested_object_fields =  ('user', 'repository', 'environment', 'milestone')
nested_array_fields = ('labels', 'assignees')
for dataset_name, documents in (('raw', raw_documents), ('cleaned', cleaned_documents)):
    wrong_nested_types = []
    for document_number, document in enumerate(documents, start=1):
        for field in nested_object_fields:
            if field in document and document[field] is not None and not isinstance(document[field], dict):
                wrong_nested_types.append((document_number, field, type(document[field]).__name__))
        for field in nested_array_fields:
            if field in document and document[field] is not None and not isinstance(document[field], list):
                wrong_nested_types.append((document_number, field, type(document[field]).__name__))
    if wrong_nested_types:
        raise TypeError(f'{dataset_name} data has unexpected nested types: {wrong_nested_types[:5]}')
    print(f'{dataset_name.title()} nested-structure check passed.')

Raw nested-structure check passed.
Cleaned nested-structure check passed.


In [17]:
# Use observed values in later queries rather than assuming a category or priority.

observed_states = sorted({d['state'] for d in cleaned_documents
    if isinstance(d.get('state'), str) and d['state'].strip()})
observed_priorities = sorted({d['priority'] for d in cleaned_documents
    if isinstance(d.get('priority'), str) and d['priority'].strip()})
observed_categories = sorted({d['issue_category'] for d in cleaned_documents
    if isinstance(d.get('issue_category'), str) and d['issue_category'].strip()})
observed_languages = sorted({d['repository']['language'] for d in cleaned_documents
    if isinstance(d.get('repository'), dict) and isinstance(d['repository'].get('language'), str)
    and d['repository']['language'].strip()})
print('Observed states:', observed_states)
print('Observed priorities:', observed_priorities)
print('Observed categories:', observed_categories)
print('Observed repository languages (first 10):', observed_languages[:10])

Observed states: ['closed', 'open']
Observed priorities: ['critical', 'high', 'low', 'medium']
Observed categories: ['bug', 'documentation', 'enhancement', 'question']
Observed repository languages (first 10): ['Go', 'Java', 'JavaScript', 'Python', 'TypeScript']


## 4.3 Creating Collections and Importing Documents


In [18]:
issues_raw = database["issues_raw"]
issues_cleaned = database["issues_cleaned"]

def inspect_id_quality(documents):
    """Count unusable and repeated IDs without removing any documents."""
    id_keys = []
    missing_ids = invalid_ids = 0
    for document in documents:
        issue_id = document.get('id')
        if 'id' not in document or issue_id is None or (isinstance(issue_id, str) and not issue_id.strip()):
            missing_ids += 1
        elif isinstance(issue_id, bool) or isinstance(issue_id, (dict, list)):
            invalid_ids += 1
        else:
            id_keys.append((type(issue_id).__name__, json.dumps(issue_id, ensure_ascii=False)))
    repeated = sum(count - 1 for count in Counter(id_keys).values() if count > 1)
    return {'missing': missing_ids, 'invalid': invalid_ids, 'duplicate_extra_records': repeated,
            'unique_id_index_supported': missing_ids == 0 and invalid_ids == 0 and repeated == 0}

raw_id_quality = inspect_id_quality(raw_documents)
cleaned_id_quality = inspect_id_quality(cleaned_documents)
print('Raw ID quality:', raw_id_quality)
print('Cleaned ID quality:', cleaned_id_quality)

Raw ID quality: {'missing': 0, 'invalid': 0, 'duplicate_extra_records': 0, 'unique_id_index_supported': True}
Cleaned ID quality: {'missing': 0, 'invalid': 0, 'duplicate_extra_records': 0, 'unique_id_index_supported': True}


In [19]:
def upsert_issue_documents(collection, documents, id_quality):
    """Idempotently replace imported records while retaining source fields."""
    if id_quality['unique_id_index_supported']:
        collection.create_index([('id', ASCENDING)], unique=True, name='id_unique')
        operations = [ReplaceOne({'id': document['id']}, document, upsert=True)
                      for document in documents]
        verification_filter = {'id': {'$in': [document['id'] for document in documents]}}
    else:
        print(f'{collection.name}: IDs are missing, invalid, or repeated; using stable import keys.')
        occurrences = Counter()
        operations = []
        import_keys = []
        for document in documents:
            canonical = json.dumps(document, sort_keys=True, ensure_ascii=False,
                                   separators=(',', ':'), allow_nan=False)
            digest = hashlib.sha256(canonical.encode('utf-8')).hexdigest()
            occurrence = occurrences[digest]
            occurrences[digest] += 1
            mongo_key = f'bugradar-import:{digest}:{occurrence}'
            stored_document = dict(document)
            stored_document['_id'] = mongo_key
            operations.append(ReplaceOne({'_id': mongo_key}, stored_document, upsert=True))
            import_keys.append(mongo_key)
        verification_filter = {'_id': {'$in': import_keys}}

    if operations:
        result = collection.bulk_write(operations, ordered=False)
        print(f'{collection.name}: {result.upserted_count} inserted/upserted, '
              f'{result.modified_count} replaced.')
    verified_count = collection.count_documents(verification_filter)
    total_count = collection.count_documents({})
    if verified_count != len(documents):
        raise RuntimeError(f'{collection.name}: verified {verified_count}, expected {len(documents)} source records.')
    print(f'{collection.name}: verified {verified_count:,} source records; total collection count {total_count:,}.')
    return total_count

In [20]:
raw_collection_count = upsert_issue_documents(issues_raw, raw_documents, raw_id_quality)
cleaned_collection_count = upsert_issue_documents(issues_cleaned, cleaned_documents, cleaned_id_quality)

issues_raw: 0 inserted/upserted, 5000 replaced.
issues_raw: verified 5,000 source records; total collection count 5,000.
issues_cleaned: 0 inserted/upserted, 5000 replaced.
issues_cleaned: verified 5,000 source records; total collection count 5,001.


## 4.4 Demonstrating Create, Read, Update, and Delete (CRUD)


In [21]:
CRUD_TEST_ID = 'BUGRADAR_NOTEBOOK_TEMP_CRUD_TEST_V1'
CRUD_TEST_MARKER = 'bugradar_mongodb_notebook_test'
previous_test = issues_cleaned.find_one({'id': CRUD_TEST_ID})
if previous_test is not None:
    if previous_test.get('test_marker') != CRUD_TEST_MARKER:
        raise RuntimeError('Reserved CRUD ID belongs to an unmarked document; refusing to overwrite it.')
    result = issues_cleaned.delete_one({'_id': previous_test['_id'], 'test_marker': CRUD_TEST_MARKER})
    if result.deleted_count != 1:
        raise RuntimeError('Could not safely remove the previous marked CRUD test document.')

crud_test_document = {
    'id': CRUD_TEST_ID,
    'title': 'Temporary BugRadar PyMongo CRUD demonstration issue',
    'issue_category': 'notebook_test',
    'state': 'open',
    'priority': 'low',
    'test_marker': CRUD_TEST_MARKER,
    'repository': {'name': 'bugradar-notebook-demo', 'language': 'Python'},
    'labels': [{'name': 'temporary-test'}],
}
crud_insert_result = issues_cleaned.insert_one(crud_test_document)
crud_created_document = issues_cleaned.find_one({'_id': crud_insert_result.inserted_id})
if crud_created_document is None or crud_created_document.get('test_marker') != CRUD_TEST_MARKER:
    raise RuntimeError('The temporary CRUD test document was not verified after insertion.')
crud_create_verified = True
print('Create verified:', crud_created_document['id'], 'with nested repository and labels.')

Create verified: BUGRADAR_NOTEBOOK_TEMP_CRUD_TEST_V1 with nested repository and labels.


In [ ]:
# READ: retrieve real issues using filters, projections, and nested-field dot notation.

read_sample = issues_cleaned.find_one(
    {'test_marker': {'$ne': CRUD_TEST_MARKER}},
    {'_id': 0, 'id': 1, 'title': 1, 'state': 1, 'priority': 1, 'repository.language': 1},
)
print('One real issue:')
pprint(read_sample, sort_dicts=False)

if observed_states:
    selected_state = observed_states[0]
    state_samples = list(issues_cleaned.find(
        {'state': selected_state, 'test_marker': {'$ne': CRUD_TEST_MARKER}},
        {'_id': 0, 'id': 1, 'title': 1, 'state': 1},
    ).limit(5))
    print(f'Up to five issues with observed state {selected_state!r}:')
    pprint(state_samples, sort_dicts=False)

if observed_languages:
    selected_language = observed_languages[0]
    nested_samples = list(issues_cleaned.find(
        {'repository.language': selected_language, 'test_marker': {'$ne': CRUD_TEST_MARKER}},
        {'_id': 0, 'id': 1, 'title': 1, 'repository.language': 1},
    ).limit(5))
    print(f'Nested query repository.language={selected_language!r}:')
    pprint(nested_samples, sort_dicts=False)
else:
    print('No string repository.language value is available for a nested-field query.')

One real issue:
{'id': 1000000,
 'title': 'Example missing for refresh the page',
 'state': 'open',
 'priority': 'medium',
 'repository': {'language': 'TypeScript'}}
Up to five issues with observed state 'closed':
[{'id': 1000001, 'title': 'Allow users to submit the form', 'state': 'closed'},
 {'id': 1000002, 'title': 'Document search configuration', 'state': 'closed'},
 {'id': 1000003, 'title': 'Add an option to save changes', 'state': 'closed'},
 {'id': 1000004, 'title': 'Allow users to filter records', 'state': 'closed'},
 {'id': 1000005, 'title': 'Unable to refresh the page', 'state': 'closed'}]
Nested query repository.language='Go':
[{'id': 1000017,
  'title': 'Unable to open a report',
  'repository': {'language': 'Go'}},
 {'id': 1000030,
  'title': 'Regression in cache',
  'repository': {'language': 'Go'}},
 {'id': 1000032,
  'title': 'CSV export fails after update',
  'repository': {'language': 'Go'}},
 {'id': 1000034,
  'title': 'Document webhook configuration',
  'repository'

In [23]:
crud_update_result = issues_cleaned.update_one(
    {'id': CRUD_TEST_ID, 'test_marker': CRUD_TEST_MARKER},
    {'$set': {'crud_demo.status': 'updated', 'crud_demo.note': 'Updated by the notebook CRUD demo'}},
)
print(f'matched_count={crud_update_result.matched_count}; modified_count={crud_update_result.modified_count}')
print('matched_count counts filter matches; modified_count counts documents whose values changed.')
crud_updated_document = issues_cleaned.find_one({'id': CRUD_TEST_ID, 'test_marker': CRUD_TEST_MARKER})
if (crud_update_result.matched_count != 1 or crud_updated_document is None
        or crud_updated_document.get('crud_demo', {}).get('status') != 'updated'):
    raise RuntimeError('The CRUD update did not persist the expected test value.')
print('Update verified: crud_demo.status is', crud_updated_document['crud_demo']['status'])
crud_update_verified = True

matched_count=1; modified_count=1
matched_count counts filter matches; modified_count counts documents whose values changed.
Update verified: crud_demo.status is updated


In [24]:
crud_delete_result = issues_cleaned.delete_one(
    {'id': CRUD_TEST_ID, 'test_marker': CRUD_TEST_MARKER},
)
crud_remaining_test = issues_cleaned.find_one({'id': CRUD_TEST_ID, 'test_marker': CRUD_TEST_MARKER})
crud_delete_verified = crud_delete_result.deleted_count == 1 and crud_remaining_test is None
if not crud_delete_verified:
    raise RuntimeError('The temporary CRUD test issue was not deleted and verified.')
print('Delete verified: only the marked temporary CRUD test issue was removed.')

Delete verified: only the marked temporary CRUD test issue was removed.


## 4.5 MongoDB Queries

In [25]:
real_issue_filter = {'test_marker': {'$ne': CRUD_TEST_MARKER}}

# 1. Count issues by each state observed in the dataset.
print('Issue counts by state:')
for state_value in observed_states:
    print(f'  {state_value}: {issues_cleaned.count_documents({**real_issue_filter, "state": state_value})}')

# 2. Closed issues with a numeric, non-negative resolution duration.
valid_closed_filter = {**real_issue_filter, 'state': 'closed',
                       'resolution_hours': {'$type': 'number', '$gte': 0}}
print('Closed issues with valid resolution_hours:', issues_cleaned.count_documents(valid_closed_filter))
pprint(list(issues_cleaned.find(valid_closed_filter,
    {'_id': 0, 'id': 1, 'title': 1, 'resolution_hours': 1}).limit(5)), sort_dicts=False)

# 3. Use 'high' if observed; otherwise query an actually observed priority.
selected_priority = 'high' if 'high' in observed_priorities else (
    observed_priorities[0] if observed_priorities else None)
if selected_priority is not None:
    priority_filter = {**real_issue_filter, 'priority': selected_priority}
    print(f'Issues with observed priority {selected_priority!r}:', issues_cleaned.count_documents(priority_filter))
    pprint(list(issues_cleaned.find(priority_filter,
        {'_id': 0, 'id': 1, 'title': 1, 'priority': 1}).limit(5)), sort_dicts=False)

# 4. Find issues belonging to an observed category.
selected_category = observed_categories[0] if observed_categories else None
if selected_category is not None:
    category_filter = {**real_issue_filter, 'issue_category': selected_category}
    print(f'Issues in observed category {selected_category!r}:', issues_cleaned.count_documents(category_filter))
    pprint(list(issues_cleaned.find(category_filter,
        {'_id': 0, 'id': 1, 'title': 1, 'issue_category': 1}).limit(5)), sort_dicts=False)

# 5. Query a nested repository field when the data contains one.
if observed_languages:
    selected_language = observed_languages[0]
    language_filter = {**real_issue_filter, 'repository.language': selected_language}
    print(f'Issues with repository.language={selected_language!r}:',
          issues_cleaned.count_documents(language_filter))

Issue counts by state:
  closed: 3901
  open: 1099
Closed issues with valid resolution_hours: 3901
[{'id': 1000001,
  'title': 'Allow users to submit the form',
  'resolution_hours': 35.34},
 {'id': 1000002,
  'title': 'Document search configuration',
  'resolution_hours': 430.48},
 {'id': 1000003,
  'title': 'Add an option to save changes',
  'resolution_hours': 837.52},
 {'id': 1000004,
  'title': 'Allow users to filter records',
  'resolution_hours': 94.02},
 {'id': 1000005,
  'title': 'Unable to refresh the page',
  'resolution_hours': 5.16}]
Issues with observed priority 'high': 904
[{'id': 1000005, 'title': 'Unable to refresh the page', 'priority': 'high'},
 {'id': 1000006, 'title': 'Update outdated filters guide', 'priority': 'high'},
 {'id': 1000010, 'title': 'How do I sync data?', 'priority': 'high'},
 {'id': 1000013,
  'title': 'report generation fails after update',
  'priority': 'high'},
 {'id': 1000030, 'title': 'Regression in cache', 'priority': 'high'}]
Issues in observe

In [26]:
# 6. Summarize issue counts by category using match, group, and sort stages.

category_pipeline = [
    {'$match': {**real_issue_filter, 'issue_category': {'$type': 'string', '$ne': ''}}},
    {'$group': {'_id': '$issue_category', 'issue_count': {'$sum': 1}}},
    {'$sort': {'issue_count': -1, '_id': 1}},
    {'$limit': 10},
]
print('Category aggregation results:')
pprint(list(issues_cleaned.aggregate(category_pipeline)), sort_dicts=False)

# 7. Compare actual collection counts with file counts.
print(f'issues_raw: {issues_raw.count_documents({})} documents; source file: {len(raw_documents)}')
print(f'issues_cleaned: {issues_cleaned.count_documents({})} documents; source file: {len(cleaned_documents)}')

Category aggregation results:
[{'_id': 'bug', 'issue_count': 2881},
 {'_id': 'enhancement', 'issue_count': 990},
 {'_id': 'documentation', 'issue_count': 665},
 {'_id': 'question', 'issue_count': 464}]
issues_raw: 5000 documents; source file: 5000
issues_cleaned: 5000 documents; source file: 5000


## 4.6 Indexing and Final Verification


In [27]:
raw_nested_summary = {
    'repository documents': issues_raw.count_documents({'repository': {'$type': 'object'}}),
    'user documents': issues_raw.count_documents({'user': {'$type': 'object'}}),
    'labels arrays': issues_raw.count_documents({'labels': {'$type': 'array'}}),
    'assignees arrays': issues_raw.count_documents({'assignees': {'$type': 'array'}}),
}
cleaned_nested_summary = {
    'repository documents': issues_cleaned.count_documents({'repository': {'$type': 'object'}}),
    'user documents': issues_cleaned.count_documents({'user': {'$type': 'object'}}),
    'labels arrays': issues_cleaned.count_documents({'labels': {'$type': 'array'}}),
    'assignees arrays': issues_cleaned.count_documents({'assignees': {'$type': 'array'}}),
}
temporary_test_record_remains = issues_cleaned.find_one({
    "id": CRUD_TEST_ID,
    "test_marker": CRUD_TEST_MARKER,
}) is not None

final_summary = {
    'database': DATABASE_NAME,
    'collections': {
        'issues_raw': {'document_count': issues_raw.count_documents({}),
                       'source_document_count': len(raw_documents),
                       'indexes': list(issues_raw.list_indexes()), 'nested_types': raw_nested_summary},
        'issues_cleaned': {'document_count': issues_cleaned.count_documents({}),
                           'source_document_count': len(cleaned_documents),
                           'indexes': list(issues_cleaned.list_indexes()), 'nested_types': cleaned_nested_summary},
    },
    'id_quality': {'raw': raw_id_quality, 'cleaned': cleaned_id_quality},
    'crud': {'create_verified': crud_create_verified,
             'update_verified': crud_update_verified,
             'delete_verified': crud_delete_verified,
             'temporary_test_record_remains': temporary_test_record_remains},
}
pprint(final_summary, sort_dicts=False, width=110)
if (not final_summary['crud']['create_verified'] or not final_summary['crud']['update_verified']
        or not final_summary['crud']['delete_verified']
        or final_summary['crud']['temporary_test_record_remains']):
    raise AssertionError('One or more CRUD verification checks did not pass.')
print('Final database and CRUD verification checks passed.')

{'database': 'bugradar_db',
 'collections': {'issues_raw': {'document_count': 5000,
                                'source_document_count': 5000,
                                'indexes': [SON([('v', 2), ('key', SON([('_id', 1)])), ('name', '_id_')]),
                                            SON([('v', 2), ('key', SON([('id', 1)])), ('name', 'id_unique'), ('unique', True)])],
                                'nested_types': {'repository documents': 5000,
                                                 'user documents': 4713,
                                                 'labels arrays': 5000,
                                                 'assignees arrays': 5000}},
                 'issues_cleaned': {'document_count': 5000,
                                    'source_document_count': 5000,
                                    'indexes': [SON([('v', 2), ('key', SON([('_id', 1)])), ('name', '_id_')]),
                                                SON([('v', 2), ('key', SON([(